# 01c · Our extensions, two-stage (statistical) training

Everything here is fit on each window's training data only, with choices made on validation, exactly like
the baselines. Outputs go to the same prediction folder, so notebook 02 evaluates them next to the paper's models.

| Name | What it is | Why |
|---|---|---|
| PCA20-Linear, PCA20-NN3 | 20 principal components of the numeric characteristics, then a linear or NN3 head | unsupervised reduction of 220 collinear characteristics |
| PLS5-NN3 | 5 PLS (SIMPLS) directions, then NN3 | reduction supervised by covariance with returns |
| LDA9-NN3 | the 9 Linear Discriminant Analysis directions separating each day's return deciles, then NN3 | reduction supervised by the decile structure the portfolio uses |
| SFA20-NN3 | the 20 slowest directions (smallest change-to-level variance) | persistence-aware: low-turnover signals, for the cost experiment |
| GMM-Experts | linear experts gated by a GMM on 10 PCA scores (K = 2-4) | different pricing rules for different kinds of firms, as in the paper's own example |
| Size-Experts | the same with hard size terciles | checks whether the GMM gate adds anything over size |
| Linear-SVR, RFF-SVR | ε-insensitive loss with L2 penalty; linear and random-Fourier-feature kernel | robust and nonlinear two-stage baselines |

The reductions are also used as starting points for E2E training (notebook 03), including decision-aware PCA.

In [ ]:
from pathlib import Path

from mlm.data import rolling_windows
from mlm.mixture import run_mixture
from mlm.reduce import run_reduced
from mlm.train import TrainConfig, run_rolling

PANEL_DIR = Path("D:/ML_Markowitz_data/panel_cn")
PRED_DIR = Path("D:/ML_Markowitz_data/preds/cn")
windows = rolling_windows("2014-01-01", "2023-12-31")
REDUCED = ["PCA20-Linear", "PCA20-NN3", "PLS5-NN3", "LDA9-NN3", "SFA20-NN3"]

In [ ]:
run_reduced(REDUCED, PANEL_DIR, windows, PRED_DIR, TrainConfig())

In [ ]:
run_mixture(PANEL_DIR, windows, PRED_DIR)

In [ ]:
svr = TrainConfig(loss="eps", eps=0.5, weight_decay=1e-4)
run_rolling(["Linear", "RFF"], PANEL_DIR, windows, PRED_DIR, svr, tag="-SVR", log_name="svr_log.csv")